# Real Estate Price Estimator

A RAG-augmented, fine-tuned GPT-4o-mini agent for real estate pricing and neighborhood recommendations.

**Pipeline:**
1. Fine-tune GPT-4o-mini on real estate Q&A data (pricing + neighborhood tasks)
2. Build a ChromaDB vector store from property listings for RAG retrieval
3. At inference: detect price-related questions → retrieve similar properties via RAG → inject context → call the fine-tuned model

In [23]:
import os
import json
import re
import time
import csv
import random
import numpy as np
import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login
from datasets import Dataset, DatasetDict, load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

load_dotenv(override=True)
openai_client = OpenAI()

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DB_PATH = os.path.join(BASE_DIR, "real_estate_vectorstore")
COLLECTION_NAME = "real_estate"
FINE_TUNE_MODEL_PREFIX = "real-estate-predictor"
BASE_MODEL = "gpt-4o-mini-2024-07-18"

In [4]:
from real_estate_item import RealEstateItem, PromptedRealEstateItem, PREFIX, QUESTION

In [21]:
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token, add_to_git_credential=True)

HF_USERNAME = "odinachidavid"  # <-- change this to your HuggingFace username
RAG_DATASET_NAME = f"{HF_USERNAME}/real_estate_properties"
QA_DATASET_NAME = f"{HF_USERNAME}/real_estate_qa"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Note: Environment variable`H

### Upload local CSVs to HuggingFace Hub (run once)

This pushes two datasets:
- **`real_estate_properties`** — the RAG knowledge base (10k property listings)
- **`real_estate_qa`** — the fine-tuning Q&A data (train / validation / test splits)

In [25]:
rag_hub = load_dataset(RAG_DATASET_NAME)
qa_hub = load_dataset(QA_DATASET_NAME)

rag_df = rag_hub["train"].to_pandas()
train_df = qa_hub["train"].to_pandas()
val_df = qa_hub["validation"].to_pandas()
test_df = qa_hub["test"].to_pandas()

print(f"RAG knowledge base: {len(rag_df):,} properties")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"\nTask types in train: {train_df['task_type'].value_counts().to_dict()}")
print(f"\nRAG columns: {list(rag_df.columns)}")
rag_df.head(3)

README.md:   0%|          | 0.00/780 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/715k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

RAG knowledge base: 10,000 properties
Train: 10,000 | Val: 10,000 | Test: 10,000

Task types in train: {'full_pricing': 2500, 'vague_pricing': 2500, 'full_neighborhood': 2500, 'single_param_neighborhood': 2500}

RAG columns: ['property_id', 'rooms', 'bathrooms', 'country', 'state', 'size', 'property_type', 'year_built', 'neighborhood_type', 'condition', 'description', 'price', 'price_range_low', 'price_range_high']


,property_id,rooms,bathrooms,country,state,size,property_type,year_built,neighborhood_type,condition,description,price,price_range_low,price_range_high
0,PROP-00001,4,2,Canada,New Brunswick,222,bungalow,2005,rural,good,"A good bungalow in a rural area, built in 2005...","$193,000","$170,000","$216,000"
1,PROP-00002,1,1,Australia,Queensland,207,cottage,1976,urban,good,"A good cottage in a urban area, built in 1976....","$213,000","$187,000","$239,000"
2,PROP-00003,6,4,UK,Wales,439,single-family home,2021,suburban,excellent,A excellent single-family home in a suburban a...,"$1,447,000","$1,273,000","$1,621,000"


## 2. Build ChromaDB Vector Store for RAG

Embed property descriptions loaded from HuggingFace using SentenceTransformer and store them in a persistent ChromaDB collection. Each document carries metadata (price, rooms, size, etc.) so we can surface them as context during inference.

In [8]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=DB_PATH)

sample_vec = encoder.encode(["sample property listing"])[0]
print(f"Embedding dimension: {sample_vec.shape[0]}")

Here are similar properties from the market for reference:

Property 1: A good villa in a rural area, built in 1987. Features include ocean views, swimming pool, basement, and private garden. Located in Queensland, Australia, this 171 sqm property offers 2 bedroom(s) and 2 bathroom(s).
  Listed at $199,000 (range: $175,000 – $223,000)

Property 2: A good villa in a rural area, built in 1998. Features include in-unit laundry, city views, walk-in closet, and ocean views. Located in Queensland, Australia, this 176 sqm property offers 1 bedroom(s) and 1 bathroom(s).
  Listed at $206,000 (range: $181,000 – $231,000)

Property 3: A excellent villa in a urban area, built in 1958. Features include city views, stainless steel appliances, fireplace, and basement. Located in Queensland, Australia, this 371 sqm property offers 6 bedroom(s) and 2 bathroom(s).
  Listed at $1,391,000 (range: $1,224,000 – $1,558,000)

Property 4: A excellent villa in a urban area, built in 1966. Features include ocean

In [ ]:
def parse_price(p):
    if pd.isna(p):
        return 0.0
    return float(str(p).replace("$", "").replace(",", ""))


existing_names = [c.name for c in chroma_client.list_collections()]

if COLLECTION_NAME not in existing_names:
    collection = chroma_client.create_collection(COLLECTION_NAME)
    BATCH = 500
    for i in tqdm(range(0, len(rag_df), BATCH), desc="Indexing properties"):
        batch = rag_df.iloc[i : i + BATCH]
        documents = batch["description"].tolist()
        vectors = encoder.encode(documents).astype(float).tolist()

        metadatas = []
        for _, row in batch.iterrows():
            metadatas.append({
                "property_id": str(row["property_id"]),
                "rooms": int(row["rooms"]),
                "bathrooms": int(row["bathrooms"]),
                "country": str(row.get("country", "")),
                "state": str(row.get("state", "")),
                "size": str(row.get("size", "")),
                "property_type": str(row.get("property_type", "")),
                "year_built": str(row.get("year_built", "")),
                "condition": str(row.get("condition", "")),
                "price": parse_price(row["price"]),
                "price_range_low": parse_price(row.get("price_range_low")),
                "price_range_high": parse_price(row.get("price_range_high")),
            })

        ids = [f"prop_{j}" for j in range(i, i + len(documents))]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

    print(f"Indexed {collection.count():,} properties into ChromaDB")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists, skipping ingestion.")

collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
print(f"Collection has {collection.count():,} documents")

In [ ]:
def find_similar_properties(query: str, n_results: int = 5) -> tuple[list[str], list[dict]]:
    """Retrieve the n most similar properties from ChromaDB for the given query."""
    vector = encoder.encode([query])
    results = collection.query(
        query_embeddings=vector.astype(float).tolist(),
        n_results=n_results,
    )
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    return documents, metadatas


def build_rag_context(documents: list[str], metadatas: list[dict]) -> str:
    """Format retrieved properties into a context string for the prompt."""
    lines = ["Here are similar properties from the market for reference:\n"]
    for i, (doc, meta) in enumerate(zip(documents, metadatas), 1):
        price = meta.get("price", 0)
        low = meta.get("price_range_low", 0)
        high = meta.get("price_range_high", 0)
        lines.append(
            f"Property {i}: {doc}\n"
            f"  Listed at ${price:,.0f} (range: ${low:,.0f} – ${high:,.0f})\n"
        )
    return "\n".join(lines)


docs, metas = find_similar_properties("3 bedroom villa in urban Queensland, Australia, good condition")
print(build_rag_context(docs, metas))

## 3. Fine-tune GPT-4o-mini

Convert the training data into OpenAI's JSONL chat format and launch a fine-tuning job. We use a subset of the training data to keep costs manageable.

In [9]:
SYSTEM_PROMPT = (
    "You are a real estate expert assistant. You help users estimate property prices "
    "and recommend neighborhoods. When given property details, provide accurate price "
    "estimates with ranges. When asked about neighborhoods, give practical location advice. "
    "Be concise but informative."
)

FINE_TUNE_SAMPLE_SIZE = 5000

def row_to_chat_messages(row):
    """Convert a train.csv row into OpenAI chat fine-tuning format."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]},
        ]
    }


train_sample = train_df.sample(n=min(FINE_TUNE_SAMPLE_SIZE, len(train_df)), random_state=42)
val_sample = val_df.sample(n=min(500, len(val_df)), random_state=42)

train_jsonl_path = os.path.join(BASE_DIR, "fine_tune_train.jsonl")
val_jsonl_path = os.path.join(BASE_DIR, "fine_tune_val.jsonl")

for path, df in [(train_jsonl_path, train_sample), (val_jsonl_path, val_sample)]:
    with open(path, "w") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row_to_chat_messages(row)) + "\n")

print(f"Wrote {len(train_sample):,} training examples to {train_jsonl_path}")
print(f"Wrote {len(val_sample):,} validation examples to {val_jsonl_path}")

with open(train_jsonl_path) as f:
    sample = json.loads(f.readline())
print(f"\nSample entry:\n{json.dumps(sample, indent=2)[:500]}...")

Wrote 5,000 training examples to /Users/Apple/vscode_projects/llm_engineering/week8/community_contributions/Odinachi/fine_tune_train.jsonl
Wrote 500 validation examples to /Users/Apple/vscode_projects/llm_engineering/week8/community_contributions/Odinachi/fine_tune_val.jsonl

Sample entry:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a real estate expert assistant. You help users estimate property prices and recommend neighborhoods. When given property details, provide accurate price estimates with ranges. When asked about neighborhoods, give practical location advice. Be concise but informative."
    },
    {
      "role": "user",
      "content": "Can you value this property for me?\n\nProperty type: townhouse\nLocation: Alberta, Canada\nNeighborhood...


In [10]:
train_file = openai_client.files.create(file=open(train_jsonl_path, "rb"), purpose="fine-tune")
val_file = openai_client.files.create(file=open(val_jsonl_path, "rb"), purpose="fine-tune")

print(f"Training file ID: {train_file.id}")
print(f"Validation file ID: {val_file.id}")

Training file ID: file-4SVRmJUHSCuBxVYb4rqYUd
Validation file ID: file-H8h4aZsCDCASSX2YKfSBPj


In [11]:
fine_tune_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=BASE_MODEL,
    suffix=FINE_TUNE_MODEL_PREFIX,
    hyperparameters={"n_epochs": 1},
)

print(f"Fine-tune job created: {fine_tune_job.id}")
print(f"Status: {fine_tune_job.status}")

Fine-tune job created: ftjob-xkh7OnookYDKnZc74prfln01
Status: validating_files


In [15]:
while True:
    job = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    print(f"Status: {job.status} | Trained tokens: {job.trained_tokens or 'N/A'}")

    if job.status in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(30)

if job.status == "succeeded":
    fine_tuned_model = job.fine_tuned_model
    print(f"\nFine-tuned model ready: {fine_tuned_model}")
else:
    print(f"\nFine-tuning ended with status: {job.status}")
    fine_tuned_model = None

Status: succeeded | Trained tokens: 891872

Fine-tuned model ready: ft:gpt-4o-mini-2024-07-18:pillowcode:real-estate-predictor:DH3aSJlR


### If you already have a fine-tuned model, set it here instead of waiting

```python
fine_tuned_model = "ft:gpt-4o-mini-2024-07-18:your-org::your-id"
```

## 4. Inference Pipeline — RAG-augmented for Price Questions

When the user asks a price-related question, we:
1. Detect the intent (pricing vs. neighborhood)
2. For pricing questions: retrieve similar properties via RAG and inject them as context
3. Call the fine-tuned model with the enriched prompt

In [16]:
PRICE_KEYWORDS = [
    "price", "cost", "worth", "value", "estimate", "market value",
    "how much", "budget", "afford", "pricing", "expensive", "cheap",
    "pay for", "listed at", "asking price",
]

def is_price_question(query: str) -> bool:
    """Heuristic check: does the user query relate to pricing/valuation?"""
    query_lower = query.lower()
    return any(kw in query_lower for kw in PRICE_KEYWORDS)


def ask_real_estate_agent(
    query: str,
    model: str | None = None,
    n_rag_results: int = 5,
    verbose: bool = True,
) -> str:
    """
    End-to-end inference:
      - Detects if the question is price-related
      - If yes, performs RAG retrieval and injects context
      - Calls the fine-tuned (or base) model
    """
    model = model or fine_tuned_model or BASE_MODEL
    is_pricing = is_price_question(query)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if is_pricing:
        docs, metas = find_similar_properties(query, n_results=n_rag_results)
        rag_context = build_rag_context(docs, metas)
        user_content = f"{query}\n\n---\n{rag_context}"
        if verbose:
            print(f"[RAG] Detected price question — retrieved {len(docs)} similar properties")
    else:
        user_content = query
        if verbose:
            print("[NO-RAG] Non-price question — calling model directly")

    messages.append({"role": "user", "content": user_content})

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_tokens=512,
    )

    answer = response.choices[0].message.content
    if verbose:
        print(f"[Model: {model}]\n")
    return answer

## 5. Test the Pipeline

In [18]:
print("=" * 80)
print("TEST 1: Full pricing question (should trigger RAG)")
print("=" * 80)
q1 = """Estimate the market value of this property.

Property type: villa
Location: Queensland, Australia
Neighborhood: urban
Rooms: 4 | Bathrooms: 3 | Size: 350 sqm
Year built: 2010 | Condition: good
Description: A good villa in an urban area, built in 2010. Features include swimming pool, granite countertops, and smart home features. Located in Queensland, Australia, this 350 sqm property offers 4 bedroom(s) and 3 bathroom(s)."""

answer1 = ask_real_estate_agent(q1)
print(answer1)

TEST 1: Full pricing question (should trigger RAG)
[RAG] Detected price question — retrieved 5 similar properties
[Model: ft:gpt-4o-mini-2024-07-18:pillowcode:real-estate-predictor:DH3aSJlR]

Based on the provided details, the estimated market value of this property is $1,152,000, with a realistic range of $1,037,000 – $1,267,000 depending on final inspection and market conditions. This reflects the good condition, urban location in Queensland, and a property size of 350 sqm.


In [19]:
print("=" * 80)
print("TEST 2: Vague pricing question (should trigger RAG)")
print("=" * 80)
q2 = "What's a 2-bedroom apartment worth in Berlin, Germany?"

answer2 = ask_real_estate_agent(q2)
print(answer2)

TEST 2: Vague pricing question (should trigger RAG)
[RAG] Detected price question — retrieved 5 similar properties
[Model: ft:gpt-4o-mini-2024-07-18:pillowcode:real-estate-predictor:DH3aSJlR]

A 2-bedroom apartment in Berlin, Germany is typically valued between $115,000 and $147,000, averaging around $131,000. Factors like condition, neighborhood, and square footage can push the price toward either end of the range.


In [ ]:
print("=" * 80)
print("TEST 3: Neighborhood question (should NOT trigger RAG)")
print("=" * 80)
q3 = "Can you suggest a good neighborhood in Ontario, Canada for a 3-bedroom townhouse?"

answer3 = ask_real_estate_agent(q3)
print(answer3)

## 6. Evaluate on Test Set

Run the pipeline on a sample from the test set and compare predicted vs. expected answers for pricing questions. We extract dollar amounts from both the predicted and expected outputs to compute error metrics.

In [20]:
def extract_price(text: str) -> float | None:
    """Pull the first dollar amount from a string like '$1,234,000'."""
    text = text.replace(",", "")
    match = re.search(r"\$?([\d]+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


EVAL_SIZE = 50
pricing_test = test_df[test_df["task_type"].str.contains("pricing")].sample(
    n=min(EVAL_SIZE, len(test_df)), random_state=42
)

errors = []
results_log = []

for idx, row in tqdm(pricing_test.iterrows(), total=len(pricing_test), desc="Evaluating"):
    predicted = ask_real_estate_agent(row["input"], verbose=False)
    expected_price = extract_price(row["output"])
    predicted_price = extract_price(predicted)

    if expected_price and predicted_price and expected_price > 0:
        pct_error = abs(predicted_price - expected_price) / expected_price * 100
        errors.append(pct_error)
        results_log.append({
            "expected": expected_price,
            "predicted": predicted_price,
            "pct_error": pct_error,
        })

if errors:
    print(f"\nEvaluated {len(errors)} pricing predictions")
    print(f"Mean Absolute % Error: {np.mean(errors):.1f}%")
    print(f"Median Absolute % Error: {np.median(errors):.1f}%")
    print(f"Within 10%: {sum(1 for e in errors if e <= 10) / len(errors) * 100:.0f}%")
    print(f"Within 20%: {sum(1 for e in errors if e <= 20) / len(errors) * 100:.0f}%")
else:
    print("No pricing predictions could be evaluated.")

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


Evaluated 50 pricing predictions
Mean Absolute % Error: 42.3%
Median Absolute % Error: 21.0%
Within 10%: 36%
Within 20%: 48%


## 7. Interactive Chat

Use the cell below to ask any real estate question. Price-related questions will automatically trigger RAG retrieval.

In [ ]:
your_question = "How much would a 5-bedroom single-family home in suburban Colorado cost? It's 400 sqm, built in 2015, in excellent condition."

answer = ask_real_estate_agent(your_question)
print(answer)

## 8. Deploy to Modal with Gradio UI

Deploy the real estate predictor as a web app on Modal. The app:
- Loads the RAG knowledge base from HuggingFace on container startup
- Builds an in-memory ChromaDB vector store with SentenceTransformer embeddings
- Serves a Gradio chat interface that routes price questions through RAG

**Prerequisites:**
1. `modal` CLI installed and authenticated (`modal setup`)
2. Modal secrets configured:
   - `my-openai-secret` with your `OPENAI_API_KEY`
   - `huggingface-secret` with your `HF_TOKEN`

Update `FINE_TUNED_MODEL` and `HF_USERNAME` in `real_estate_app.py` if yours differ.

In [ ]:
!modal deploy real_estate_app.py

### For quick testing (ephemeral deploy — stops when you close the terminal):
```bash
modal serve real_estate_app.py
```

### For persistent deployment:
```bash
modal deploy real_estate_app.py
```

After deployment, Modal will print a URL like `https://your-username--real-estate-predictor-web.modal.run` — open it to use the Gradio chat UI.

In [ ]:
import modal

Predictor = modal.Cls.from_name("real-estate-predictor", "RealEstatePredictor")
predictor = Predictor()

result = predictor.predict.remote("How much is a 3-bedroom villa worth in urban Queensland, Australia?")
print(f"Used RAG: {result['used_rag']}")
print(f"\nAnswer:\n{result['answer']}")

if result["rag_context"]:
    print(f"\nRAG Context:\n{result['rag_context'][:500]}...")